In [1]:
# Importing libraries I will need
import xarray as xr
import pandas as pd
import PyCO2SYS as pyco2

In [2]:
# Load dataset
file_path = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/CSIR_ML6_Merged.nc"
ds = xr.open_dataset(file_path)
df = ds.to_dataframe().reset_index()

In [3]:
# Drop unwanted columns
df = df.drop(columns=["latitude_merc", "longitude_merc"], errors="ignore")

In [4]:
# Rename columns
rename_map = {
    "time": "Date",
    "latitude": "Lat",
    "longitude": "Longitude",
    "fgco2": "fgCO2",
    "pCO2": "pCO2",
    "depth": "DEPTH(M)",
    "so": "Salinity",
    "thetao": "Temperature",
    "AT": "Total Alkalinity"
}
df = df.rename(columns=rename_map)

In [5]:
# Ensure Date is datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day

In [6]:
# Add Season
def classify_season(month):
    if month in [12, 1, 2]:
        return "Summer"
    elif month in [3, 4, 5]:
        return "Autumn"
    elif month in [6, 7, 8]:
        return "Winter"
    elif month in [9, 10, 11]:
        return "Spring"
df["Season"] = df["Month"].apply(classify_season)

In [7]:
# Computing unknown carbonate components
def compute_carbonate_system(row):
    results = pyco2.sys(
        par1=row["pCO2"],                    # input pCO2
        par2=row["Total Alkalinity"],        # TA
        par1_type=4,                         # 4 = pCO2
        par2_type=1,                         # 1 = TA
        salinity=row["Salinity"],            # Salinity
        temperature=row["Temperature"],      # Temperature (°C)
        pressure=0,                          # Surface pressure
        opt_pH_scale=1,                      # Total scale
        opt_k_carbonic=14,                   # Southern Ocean constants
        opt_k_bisulfate=2                    # Southern Ocean constants
    )
    return pd.Series({
        "Total Carbon": results["dic"],       # Dissolved Inorganic Carbon
        "pH": results["pH"],                  # pH on total scale
        "OmegaAragonite": results["saturation_aragonite"]  # Saturation state
    })
   
df[["Total Carbon", "pH", "OmegaAragonite"]] = df.apply(compute_carbonate_system, axis=1)

In [8]:
# --- Save back to NetCDF ---
# Convert DataFrame to xarray Dataset
ds_out = df.set_index(["Date", "Lat", "Longitude"]).to_xarray()


In [9]:
# Save to NetCDF file
output_path = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/CSIR_ML6_Final.nc"
ds_out.to_netcdf(output_path)

print(f"Final dataset saved to: {output_path}")



Final dataset saved to: /Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/CSIR_ML6_Final.nc


In [10]:
print(df.columns.tolist())


['Date', 'Lat', 'Longitude', 'fgCO2', 'pCO2', 'DEPTH(M)', 'Salinity', 'Temperature', 'Total Alkalinity', 'Year', 'Month', 'Day', 'Season', 'Total Carbon', 'pH', 'OmegaAragonite']


In [11]:
print(df.head(50))

         Date   Lat  Longitude         fgCO2        pCO2  DEPTH(M)   Salinity  \
0  2009-12-01 -69.5       15.5  1.257129e-09  379.237760  3.819495  32.952362   
1  2009-12-01 -69.5       16.5  1.625722e-09  379.463568  3.819495  33.324688   
2  2009-12-01 -69.5       17.5  1.083595e-09  374.479944  3.819495  33.153782   
3  2009-12-01 -69.5       18.5 -1.756832e-09  367.623003  3.819495  33.367413   
4  2009-12-01 -69.5       19.5 -1.511893e-09  367.727819  3.819495  33.584095   
5  2009-12-01 -69.5       20.5 -5.565730e-09  355.119041  3.819495  33.602406   
6  2009-12-01 -69.5       21.5 -5.282002e-09  356.758551  3.819495  33.686329   
7  2009-12-01 -69.5       22.5 -8.171611e-09  350.715864  3.819495  33.712273   
8  2009-12-01 -69.5       23.5 -5.698753e-09  352.820147  3.819495  33.808403   
9  2009-12-01 -69.5       24.5 -2.150964e-09  361.551652  3.819495  33.980835   
10 2009-12-01 -68.5       15.5  5.516835e-09  387.786320  3.819495  33.904537   
11 2009-12-01 -68.5       16